# Aula 06 - Aprendizado Não Supervisionado parte I

**Módulo 03 IN** - Lógica para predição com inteligência artificial
**01/09/2026 - Sprint 3 - Prof. Ovidio Lopes da Cruz Netto**

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/canaldoovidio/2026-2A-M03/blob/main/notebooks/aula06.ipynb)

## O que este notebook é

É o primeiro contato do case com aprendizado sem rótulo. As Aulas 04 e 05 previram um número
(a produção de frango) a partir de entradas conhecidas. Aqui não existe alvo: o K-means recebe os
cinco produtos em cada trimestre e agrupa sozinho, sem que ninguém diga o que é "parecido".

O notebook roda a mesma técnica duas vezes sobre os cinco produtos, mudando apenas a pergunta
que os dados respondem, e mede um resultado contra-intuitivo: a leitura que a silhueta aponta
como melhor separação é justamente a que não recupera o calendário. Os Atos 1 e 2 mostram, cada
um à sua vez, o que está por trás desse número.

## Ao final deste notebook você terá

1. confirmado o modelo da Aula 05 rodando do zero, em uma única célula;
2. calculado a distância euclidiana entre trimestres em variáveis padronizadas;
3. agrupado os cinco produtos em nível com `KMeans` e medido a concordância com o calendário;
4. repetido o agrupamento sobre a participação anual e visto a concordância saltar para 98,3%;
5. lido o perfil sazonal de cada produto e identificado as duas exceções de 2008;
6. testado, no desafio, o que muda ao incluir um ano incompleto ou ao reduzir K.


## 1. Retomada: da Aula 05 ao MAPE de 1,60%

Este bloco é para quem chega com o ambiente local ainda não funcionando. A célula abaixo é
**autocontida**: lê os cinco CSVs, reconstrói a base analítica da Aula 04 (as defasagens de 1 e de
4 trimestres e o par seno/cosseno de sazonalidade), ajusta a regressão da Aula 05 e imprime o MAPE
do modelo contra a baseline de coeficiente fixo. Nenhuma célula anterior precisa ter rodado antes
dela.

Só `pandas`, `numpy` e `scikit-learn`, que o Google Colab já traz instalados. Rodar esta célula é
o objetivo do bloco: se o número final for MAPE do modelo 1,60% e da baseline 1,69%, o ambiente
está pronto para o resto da aula.


In [ ]:
import os
import urllib.request

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.preprocessing import StandardScaler

SERIES = [
    "abate_bovinos",
    "abate_suinos",
    "abate_frangos",
    "producao_ovos",
    "producao_leite",
]
ALVO = "abate_frangos"

# mesma resolucao de caminho das aulas anteriores: funciona no repositorio
# clonado (CSVs em ../dados/) e no Colab (baixa da versao publicada)
BASE_LOCAL = os.path.join("..", "dados")
BASE_BRUTA = ("https://raw.githubusercontent.com/canaldoovidio/2026-2A-M03/"
              "main/dados/")

caminhos = {}
for nome in SERIES:
    arquivo = nome + ".csv"
    local = os.path.join(BASE_LOCAL, arquivo)
    if os.path.exists(local):
        caminhos[nome] = local
    else:
        if not os.path.exists(arquivo):
            try:
                urllib.request.urlretrieve(BASE_BRUTA + arquivo, arquivo)
            except Exception as erro:
                raise RuntimeError(
                    "Nao foi possivel baixar '%s' pela internet (%s). "
                    "Se a rede da sala falhou, peca a pasta 'dados' para uma dupla "
                    "que tenha o repositorio clonado no computador (ela fica na raiz "
                    "do repositorio) e coloque essa pasta ao lado deste notebook. "
                    "Depois, rode esta celula de novo." % (arquivo, erro)
                ) from erro
        caminhos[nome] = arquivo

# junta as cinco series por periodo (interseccao, mesma regra da Aula 04)
base = None
for nome in SERIES:
    coluna = (pd.read_csv(caminhos[nome])[["periodo", "valor"]]
              .rename(columns={"valor": nome}))
    base = coluna if base is None else base.merge(coluna, on="periodo", how="inner")
base = base.sort_values("periodo").reset_index(drop=True)
base["trimestre"] = base["periodo"].str[-1].astype(int)

# defasagens da Aula 04: o passado entra na linha do presente
base["frangos_lag1"] = base[ALVO].shift(1)
base["frangos_lag4"] = base[ALVO].shift(4)

# sazonalidade como par seno/cosseno (Aula 04)
base["sen"] = np.sin(2 * np.pi * base["trimestre"] / 4)
base["cos"] = np.cos(2 * np.pi * base["trimestre"] / 4)

analitica = base.dropna().reset_index(drop=True)
print("base analitica: %d linhas, de %s a %s"
      % (len(analitica), analitica["periodo"].iloc[0], analitica["periodo"].iloc[-1]))

# corte por data (Aula 05): oito trimestres reservados para teste
FEATURES = ["frangos_lag1", "frangos_lag4", "sen", "cos"]
N_TESTE = 8
corte = len(analitica) - N_TESTE
treino, teste = analitica.iloc[:corte], analitica.iloc[corte:]
print("treino: %d linhas   teste: %d linhas" % (len(treino), len(teste)))

# padroniza so com o treino, ajusta a regressao e preve o teste
escalador = StandardScaler().fit(treino[FEATURES].to_numpy(float))
modelo = LinearRegression().fit(
    escalador.transform(treino[FEATURES].to_numpy(float)), treino[ALVO].to_numpy(float))
previsto = modelo.predict(escalador.transform(teste[FEATURES].to_numpy(float)))
y_teste = teste[ALVO].to_numpy(float)
mape_modelo = mean_absolute_percentage_error(y_teste, previsto) * 100

# baseline da Aula 05: ano anterior vezes um fator fixo, estimado so no treino
fator = float((treino[ALVO] / treino["frangos_lag4"]).mean())
baseline = teste["frangos_lag4"].to_numpy(float) * fator
mape_baseline = mean_absolute_percentage_error(y_teste, baseline) * 100

print()
print("MAPE do modelo (regressao linear):        %.2f%%" % mape_modelo)
print("MAPE da baseline (coeficiente fixo):       %.2f%%" % mape_baseline)


## 2. O que o K-means faz

Todo modelo até aqui recebia um rótulo para aprender: o valor de `abate_frangos` no trimestre
seguinte. O K-means não recebe rótulo nenhum. Ele recebe só as observações e um número `K`, e
devolve `K` grupos, escolhidos para que observações do mesmo grupo fiquem próximas entre si e
observações de grupos diferentes fiquem distantes.

"Próximas" tem uma definição precisa: distância euclidiana, a mesma raiz da soma dos quadrados das
diferenças que se usa em duas dimensões, estendida para as cinco colunas de produção. Como as
cinco séries têm escalas muito diferentes (leite em toneladas, ovos em dúzias, abates em
quilogramas), a distância só faz sentido depois de padronizar cada coluna, exatamente como a
`StandardScaler` já fez para a regressão da seção 1.

A célula abaixo mede essa distância entre três trimestres, todos com as cinco colunas em nível
padronizadas.


In [ ]:
tabelas = {}
for nome in SERIES:
    df = pd.read_csv(caminhos[nome])
    tabelas[nome] = dict(zip(df["periodo"], df["valor"].astype(float)))

# interseccao dos periodos das cinco series, ordenada no tempo
periodos = sorted(set.intersection(*[set(t) for t in tabelas.values()]))
X = np.array([[tabelas[s][p] for s in SERIES] for p in periodos])
print("base dos cinco produtos: %d trimestres, de %s a %s"
      % (len(periodos), periodos[0], periodos[-1]))

X_padronizado = StandardScaler().fit_transform(X)

par_perto = ("1997-T1", "2011-T3")
par_longe = ("1997-T1", "2025-T4")

for a, b in (par_perto, par_longe):
    ia, ib = periodos.index(a), periodos.index(b)
    distancia = np.linalg.norm(X_padronizado[ia] - X_padronizado[ib])
    print("distancia euclidiana entre %s e %s: %.3f" % (a, b, distancia))


`2011-T3` fica mais perto de `1997-T1` do que `2025-T4` fica, mesmo os três sendo trimestres
distintos do calendário (T1, T3 e T4). O que aproxima ou afasta dois trimestres, nessa conta, é o
**nível** de produção das cinco séries, e nível é dominado por quanto tempo passou, porque as
cinco séries cresceram ao longo das quase três décadas. A seção 3 agrupa exatamente essa
grandeza, com `K=4`, e mostra o que esse domínio da tendência produz.

## 3. Ato 1: agrupar os níveis de produção

`KMeans(n_clusters=4, n_init=50, random_state=42)` roda o algoritmo 50 vezes com pontos de partida
diferentes e mantém a melhor, o suficiente para não depender de uma inicialização de sorte.
`random_state=42` fixa a sequência de sorteios, para que a célula devolva sempre o mesmo resultado.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

K = 4
SEMENTE = 42

anos = np.array([int(p[:4]) for p in periodos])
tris = np.array([int(p[-1]) for p in periodos])

rotulos1 = KMeans(n_clusters=K, n_init=50, random_state=SEMENTE).fit_predict(X_padronizado)
silhueta1 = silhouette_score(X_padronizado, rotulos1)


def concordancia(rotulos, tris):
    '''Fracao das linhas cobertas pelo trimestre majoritario de cada cluster.'''
    acertos = 0
    for c in set(rotulos):
        do_cluster = tris[rotulos == c]
        acertos += max((do_cluster == t).sum() for t in (1, 2, 3, 4))
    return acertos / len(tris)


conc1 = concordancia(rotulos1, tris)

# ordena os clusters pela posicao media no tempo, do mais antigo ao mais recente
ordem = sorted(set(rotulos1), key=lambda c: np.mean(np.where(rotulos1 == c)[0]))
print("intervalo de cada cluster (ordenado no tempo):")
for i, c in enumerate(ordem):
    idx = np.where(rotulos1 == c)[0]
    print("  epoca %d: %s a %s  (%d trimestres)"
          % (i + 1, periodos[idx.min()], periodos[idx.max()], len(idx)))

print()
print("silhueta: %.4f" % silhueta1)
print("concordancia com o trimestre do calendario: %.1f%%" % (conc1 * 100))
print("acaso, com quatro grupos e quatro trimestres: 25.0%")


Os quatro clusters formam intervalos contíguos do tempo: o primeiro cobre 1997 a 2004, o último
cobre 2019 a 2026, cada um reunindo anos vizinhos entre si. A concordância com o trimestre do
calendário fica perto de 26,5%, próxima dos 25% que o acaso já entregaria sozinho com quatro grupos
e quatro trimestres possíveis. O K-means encontrou uma estrutura real nos dados: o crescimento ao
longo do tempo domina a distância entre os pontos padronizados, e é essa estrutura que os quatro
clusters capturam.

## 4. Ato 2: agrupar a participação de cada trimestre no ano

Para separar o crescimento de longo prazo do padrão sazonal, cada valor vira a fração que ele
representa no total do próprio ano: `producao_leite` de um trimestre dividido pela soma dos quatro
trimestres daquele mesmo ano, para cada uma das cinco séries. Um ano com os quatro trimestres
medidos tem suas quatro frações somando exatamente 1, em qualquer nível de produção.

2026 entra na base com um único trimestre (T1) até a data de corte dos dados. Incluí-lo faria esse
trimestre valer sozinho 100% do "ano", um artefato da ausência dos outros três trimestres. Por
isso só os anos completos participam, e a base cai de 117 para 116 linhas.


In [ ]:
completos = {a for a in set(anos.tolist()) if (anos == a).sum() == 4}
mascara = np.array([a in completos for a in anos])

X_completos, anos2, tris2 = X[mascara], anos[mascara], tris[mascara]
periodos2 = [p for p, m in zip(periodos, mascara) if m]

participacao = np.empty_like(X_completos)
for ano in completos:
    linhas = anos2 == ano
    participacao[linhas] = X_completos[linhas] / X_completos[linhas].sum(axis=0)

print("base da participacao anual: %d linhas, de %s a %s"
      % (len(periodos2), periodos2[0], periodos2[-1]))

participacao_padronizada = StandardScaler().fit_transform(participacao)
rotulos2 = KMeans(n_clusters=K, n_init=50, random_state=SEMENTE).fit_predict(
    participacao_padronizada)
silhueta2 = silhouette_score(participacao_padronizada, rotulos2)
conc2 = concordancia(rotulos2, tris2)

print("silhueta: %.4f" % silhueta2)
print("concordancia com o trimestre do calendario: %.1f%%" % (conc2 * 100))
print()
print("comparacao dos dois atos:")
print("  ato 1 (niveis):        silhueta %.4f, concordancia %.1f%%" % (silhueta1, conc1 * 100))
print("  ato 2 (participacao):  silhueta %.4f, concordancia %.1f%%" % (silhueta2, conc2 * 100))


A concordância salta de 26,5% para 98,3%: quase todo trimestre cai no cluster que corresponde ao seu
próprio trimestre do calendário. A silhueta faz o caminho inverso, e cai de 0,4795 para 0,2853.

**A silhueta mede separação geométrica entre os clusters, uma propriedade distinta da utilidade
para o case.** O ato 1 tem silhueta mais alta porque décadas inteiras de crescimento produzem
grupos bem separados no
espaço padronizado. O ato 2 responde à pergunta que interessa (que trimestre é este, do ponto de
vista de demanda de ração) e tem silhueta mais baixa, porque a diferença entre "um pouco mais no
terceiro trimestre" e "um pouco menos no primeiro" é sutil por natureza: a produção nunca some num
trimestre e dobra no seguinte. Escolher K, ou julgar um agrupamento, só pela silhueta teria
descartado o resultado que serve ao Modelo 2 do TAPI.

## 5. Interpretação: o perfil sazonal e as exceções de 2008

A tabela abaixo é a participação média de cada trimestre no ano, por produto. Ela é o que o
cluster do ato 2 está, na prática, recuperando.


In [ ]:
mapa_cluster_para_trimestre = {}
for c in set(rotulos2):
    do_cluster = tris2[rotulos2 == c]
    mapa_cluster_para_trimestre[c] = max((1, 2, 3, 4), key=lambda t: (do_cluster == t).sum())

medias = np.array([participacao[tris2 == t].mean(axis=0) for t in (1, 2, 3, 4)]) * 100

print("%-6s" % "", end="")
for nome in SERIES:
    print("%16s" % nome, end="")
print()
for i, t in enumerate((1, 2, 3, 4)):
    print("T%-5d" % t, end="")
    for j in range(len(SERIES)):
        print("%15.2f%%" % medias[i, j], end="")
    print()

amplitude = medias.max(axis=0) - medias.min(axis=0)
print()
print("amplitude sazonal (maior menos menor trimestre, em pontos percentuais):")
for nome, amp in sorted(zip(SERIES, amplitude), key=lambda par: -par[1]):
    print("  %-16s %.2f p.p." % (nome, amp))

fora = [p for p, r, t in zip(periodos2, rotulos2, tris2) if mapa_cluster_para_trimestre[r] != t]
print()
print("trimestres que caem fora do proprio grupo: %s" % fora)


`producao_leite` tem a maior amplitude sazonal (perto de 3,85 pontos percentuais) e pico no
quarto trimestre. As três carnes (bovinos, suínos e frangos) têm pico no terceiro trimestre e
amplitude bem menor, entre 1,10 e 2,35 pontos percentuais. `producao_ovos` fica perto do abate de
frangos, com a menor amplitude entre as cinco.

As duas exceções do ato 2 caem em 2008, o ano da crise financeira internacional. Cada linha de
`producao_leite`, `producao_ovos` e das três carnes entra na conta igualmente, e uma única
observação atípica desloca a fração dos outros trimestres do mesmo ano junto: se um trimestre de
2008 produziu menos do que o padrão da série, os outros três trimestres daquele ano automaticamente
ficam com uma fatia maior do total anual, mesmo sem terem produzido mais em termos absolutos.

## 6. Desafio

Duas perguntas para responder rodando código, sem consultar o material de apoio antes de tentar.
As respostas estão registradas lá, para conferência depois.

1. A seção 4 descarta 2026 por ter um único trimestre medido. Refaça o cálculo de participação
   **incluindo** 2026-T1 como se o ano estivesse completo (ou seja, sem a máscara de anos
   completos) e rode o K-means de novo sobre esse resultado. O que muda na concordância e por quê?
2. Repita o ato 2 (participação anual) com `n_clusters=2` em vez de 4. O que os dois clusters
   passam a representar, e o que se perde em relação aos quatro trimestres do ato 2 original?


In [ ]:
# 1. participacao incluindo 2026 como se o ano estivesse completo
# dica: repita a conta da secao 4 sem filtrar por "completos"


# 2. ato 2 com K=2
# dica: reaproveite "participacao_padronizada" e troque o n_clusters do KMeans


resposta_1 = "..."
resposta_2 = "..."
